In [1]:
"""
Convert the .npz action/style dataset into the same .mat layout used by
BaseTwoModalDataset (Xray / OfficeHome / Office31 style):

  - one data file per "domain" (here: per style), containing:
        'labels'        : (1, N) int array of action-class ids
        FEATURE_KEY     : (N, D, 1, 1) float32 feature array
  - one split file containing:
        'targetDomain_splitFlag'   : (1, NUM_TRIALS) object array
                                      -> (1, NUM_DOMAINS) object array
                                      -> (1, N_domain) uint8 array, values
                                         0 = unused, 1 = train, 2 = test
        'targetDomain_unseenClass' : same nesting, leaf = (1, NUM_CLASSES)
                                      uint8 indicator of which action
                                      classes are "unseen" for that trial
                                      (same indicator repeated for every
                                      domain in that trial, matching how
                                      the original split files store it)

Run this where `inpDataset_unique_char.npz` is reachable, then point your
BaseTwoModalDataset config block at the OUT_DIR / DOMAIN_SET / DATASET_DETAILS
printed at the end.
"""


'\nConvert the .npz action/style dataset into the same .mat layout used by\nBaseTwoModalDataset (Xray / OfficeHome / Office31 style):\n\n  - one data file per "domain" (here: per style), containing:\n        \'labels\'        : (1, N) int array of action-class ids\n        FEATURE_KEY     : (N, D, 1, 1) float32 feature array\n  - one split file containing:\n        \'targetDomain_splitFlag\'   : (1, NUM_TRIALS) object array\n                                      -> (1, NUM_DOMAINS) object array\n                                      -> (1, N_domain) uint8 array, values\n                                         0 = unused, 1 = train, 2 = test\n        \'targetDomain_unseenClass\' : same nesting, leaf = (1, NUM_CLASSES)\n                                      uint8 indicator of which action\n                                      classes are "unseen" for that trial\n                                      (same indicator repeated for every\n                                      domain in tha

In [2]:
import os
import numpy as np
import scipy.io


/tmp/ipykernel_76354/3186888419.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy.io


In [ ]:
# ----------------------------------------------------------------------
# Config -- tweak these as needed
# ----------------------------------------------------------------------
NPZ_PATH = "./inpDataset.npz"
OUT_DIR = "./ActionStyleDataset/"

In [ ]:
# STYLE_LABELS = [
#     "angry", "childlike", "depressed", "neutral",
#     "old", "proud", "sexy", "strutting",
# ]  # all 8 styles -> treated as domains
STYLE_LABELS = [
    "angry", "childlike", "depressed", "neutral",
    "old", "proud", "strutting",
]  # all 8 styles -> treated as domains -> 7 domain (remove sexy, it has no sample for some classes)


PREFIX = "ActionStyle-"
SUFFIX = "-clip.mat"
FEATURE_KEY = "clip_features"
SPLIT_FILE_NAME = "instanceSplit_actionStyle_unseen2.mat"

In [5]:
# ACTION_LABELS = ["punch", "jump", "kick", "run", "walk"]
ACTION_LABELS = ["punch", "jump", "kick", "walk"]

ACTION_FULL_NAMES = [
    "walk_s1", "walk_s2",
    "walk_lturn_s1", "walk_rturn_s1", "walk_lturn_l1", "walk_lturn_l2",
    "walk_l1", "walk_l2",
    "walk_rturn_s2", "walk_lturn_s2", "walk_rturn_l1", "walk_lturn_l3",
    "run", "run_lturn", "run_rturn",
    "jump_1", "jump_2",
    "punch_r", "punch_l", "punch_qr", "punch_ql",
    "kick_l", "kick_r",
    "trans_jump2walk", "trans_walk2jump", "trans_punch2kick",
    "trans_walk2punch", "trans_run2jump",
]

In [6]:
NUM_TRIALS = 6
NUM_UNSEEN_ACTIONS = 2          # held out per trial, same across all domains
TRAIN_RATIO = 0.7               # of the *seen*-class instances per domain
SEED = 42

# ----------------------------------------------------------------------
rng = np.random.default_rng(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

# ---- load npz, derive style/action labels exactly like ActionStyleDataset ----
data = np.load(NPZ_PATH, allow_pickle=True)
x = data["clip_arrays"]            # (537, 1, 512)
names = data["names"]              # e.g. "Abe_childlike_05_001"

In [ ]:
styles = np.array([n.split("_")[1] for n in names])
action_idx = np.array([n.split("_")[2] for n in names])
actions = np.array(
    [ACTION_FULL_NAMES[int(i) - 1].split("_")[0] for i in action_idx]
)

# drop transition clips (trans_*), keep every style this time
keep = actions != "trans"
x, styles, actions = x[keep], styles[keep], actions[keep]

# we used sampleing and it hurts differences between walk and run
# so we ignore run class
keep = actions != "run"
x, styles, actions = x[keep], styles[keep], actions[keep]

# balance dataset
keep = (rng.random((actions != "walk").shape) > 0.7) + (actions != "walk")
x, styles, actions = x[keep], styles[keep], actions[keep]

keep = styles != "sexy"
x, styles, actions = x[keep], styles[keep], actions[keep]


In [8]:
action_label_to_id = {label: i for i, label in enumerate(ACTION_LABELS)}
action_ids = np.array([action_label_to_id[a] for a in actions])

In [9]:
action_label_to_id

{'punch': 0, 'jump': 1, 'kick': 2, 'walk': 3}

In [10]:
feat = x.reshape(x.shape[0], -1).astype(np.float32)  # (N, D)

In [11]:
from collections import Counter
# ---- write one data .mat file per style/domain ----
print("Per-style sample counts:")
for style in STYLE_LABELS:
    mask = styles == style
    feat_s = feat[mask]
    labels_s = action_ids[mask]

    mat_dict = {
        "labels": labels_s.reshape(1, -1).astype(np.int64),
        FEATURE_KEY: feat_s.reshape(feat_s.shape[0], feat_s.shape[1], 1, 1),
    }
    print(Counter(labels_s.tolist()).most_common())
    out_path = os.path.join(OUT_DIR, f"{PREFIX}{style}{SUFFIX}")
    scipy.io.savemat(out_path, mat_dict)
    print(f"  {style:10s} -> {feat_s.shape[0]:4d} samples  ({out_path})")

Per-style sample counts:
[(0, 35), (3, 34), (1, 24), (2, 24)]
  angry      ->  117 samples  (./ActionStyleDataset/ActionStyle-angry-clip.mat)
[(0, 37), (2, 27), (3, 23), (1, 23)]
  childlike  ->  110 samples  (./ActionStyleDataset/ActionStyle-childlike-clip.mat)
[(3, 46), (0, 41), (2, 28), (1, 18)]
  depressed  ->  133 samples  (./ActionStyleDataset/ActionStyle-depressed-clip.mat)
[(0, 48), (3, 27), (1, 23), (2, 19)]
  neutral    ->  117 samples  (./ActionStyleDataset/ActionStyle-neutral-clip.mat)
[(3, 32), (2, 27), (0, 23), (1, 13)]
  old        ->   95 samples  (./ActionStyleDataset/ActionStyle-old-clip.mat)
[(3, 38), (0, 22), (1, 20), (2, 11)]
  proud      ->   91 samples  (./ActionStyleDataset/ActionStyle-proud-clip.mat)
[(3, 47), (0, 34), (2, 23), (1, 7)]
  strutting  ->  111 samples  (./ActionStyleDataset/ActionStyle-strutting-clip.mat)


In [12]:
# ---- build the split file ----
num_domains = len(STYLE_LABELS)
num_classes = len(ACTION_LABELS)

splitFlag_cells = np.empty((1, NUM_TRIALS), dtype=object)
unseenClass_cells = np.empty((1, NUM_TRIALS), dtype=object)

In [13]:
# for unifor unseen class
used_list = []
for t in range(NUM_TRIALS):
    unseen_action_ids = rng.choice(num_classes, size=NUM_UNSEEN_ACTIONS, replace=False)
    unseen_action_ids.sort()
    while True:
        rejected = False
        for arr in used_list:
            if np.all(unseen_action_ids == arr):
                unseen_action_ids = rng.choice(num_classes, size=NUM_UNSEEN_ACTIONS, replace=False)
                unseen_action_ids.sort()
                rejected = True
        if not rejected:
            used_list.append(unseen_action_ids)
            break
    print(used_list)


    unseen_indicator = np.zeros(num_classes, dtype=np.uint8)
    unseen_indicator[unseen_action_ids] = 1

    domain_splitFlags = np.empty((1, num_domains), dtype=object)
    domain_unseen = np.empty((1, num_domains), dtype=object)

    for d, style in enumerate(STYLE_LABELS):
        mask = styles == style
        labels_s = action_ids[mask]
        n = labels_s.shape[0]

        flag = np.zeros(n, dtype=np.uint8)
        is_unseen = np.isin(labels_s, unseen_action_ids)

        # unseen-class instances are never trained on -> test only
        flag[is_unseen] = 2

        # seen-class instances: split train/test per TRAIN_RATIO
        seen_idx = np.where(~is_unseen)[0]
        rng.shuffle(seen_idx)
        n_train = int(len(seen_idx) * TRAIN_RATIO)
        flag[seen_idx[:n_train]] = 1
        flag[seen_idx[n_train:]] = 2

        domain_splitFlags[0, d] = flag.reshape(1, -1)
        domain_unseen[0, d] = unseen_indicator.reshape(1, -1)

    splitFlag_cells[0, t] = domain_splitFlags
    unseenClass_cells[0, t] = domain_unseen

[array([0, 2])]
[array([0, 2]), array([1, 3])]
[array([0, 2]), array([1, 3]), array([2, 3])]
[array([0, 2]), array([1, 3]), array([2, 3]), array([0, 3])]
[array([0, 2]), array([1, 3]), array([2, 3]), array([0, 3]), array([0, 1])]
[array([0, 2]), array([1, 3]), array([2, 3]), array([0, 3]), array([0, 1]), array([1, 2])]


In [14]:

split_path = os.path.join(OUT_DIR, SPLIT_FILE_NAME)
scipy.io.savemat(
    split_path,
    {
        "targetDomain_splitFlag": splitFlag_cells,
        "targetDomain_unseenClass": unseenClass_cells,
    },
)
print(f"\nwrote split file -> {split_path}")

# ----------------------------------------------------------------------
# Config block to paste into your existing notebook, to use with
# BaseTwoModalDataset unchanged:
# ----------------------------------------------------------------------
print(
    "\n--- use with BaseTwoModalDataset ---\n"
    f"DOMAIN_SET = {STYLE_LABELS}\n"
    f"DATA_DIR = '{OUT_DIR}'\n"
    "DATASET_DETAILS = {\n"
    f"    'prefix': '{PREFIX}',\n"
    f"    'suffix': '{SUFFIX}',\n"
    f"    'resnet_feature': '{FEATURE_KEY}',\n"
    f"    'split_file_name': '{SPLIT_FILE_NAME}',\n"
    "}"
)


wrote split file -> ./ActionStyleDataset/instanceSplit_actionStyle_unseen2.mat

--- use with BaseTwoModalDataset ---
DOMAIN_SET = ['angry', 'childlike', 'depressed', 'neutral', 'old', 'proud', 'strutting']
DATA_DIR = './ActionStyleDataset/'
DATASET_DETAILS = {
    'prefix': 'ActionStyle-',
    'suffix': '-clip.mat',
    'resnet_feature': 'clip_features',
    'split_file_name': 'instanceSplit_actionStyle_unseen2.mat',
}
